# Tutorial 3 — Integrating the FIBO Ontology into the Vanilla LLM and the Agent

Tutorial 1 ended at a `backend.complete(system, user)` call. The system prompt was hand-written.

Tutorial 2 ended at a `Session.add` / `ask` / `run` call. The agent's tools were ontology-aware *because the ontology was passed in* — but we used a placeholder schema.

This tutorial does the integration. The same FIBO Business Entities ontology drives:

- **Section A — Vanilla LLM + ontology.** Inject `ontology.to_extraction_context()` into the Tutorial 1 `complete()` call. The hand-written system prompt becomes a generated, schema-grounded one.
- **Section B — Agent + ontology.** Pass the same ontology to `Seocho.local(ontology=...)`; every tool call (extract, validate, write) inherits the FIBO schema and the indexing pipeline stamps `context_hash` on every output.
- **Section C — Evolution and provenance.** Merge FIBO + a non-FIBO `Product` extension into v2; show the `context_hash` divergence, the `migration_plan`, and the `ontology_identity_hash` stamped on the inferred `RuleSet` (the same hash promotion gates check before applying rules).

**FIBO source.** Authentic FIBO is at <https://spec.edmcouncil.org/fibo/> (MIT). The bundled `examples/datasets/fibo_be_minimal.ttl` is a teaching subset — five Business Entities classes (`Party → Person | LegalPerson → FormalOrganization → Corporation`) and a handful of properties, with `skos:exactMatch` links back to canonical FIBO IRIs.

Source references — `seocho/ontology.py` (`Ontology`, `NodeDef`, `RelDef`, `merge`, `migration_plan`), `seocho/ontology_context.py` (`compile_ontology_context`), `seocho/index/pipeline.py` (`IndexingPipeline`, `IndexingResult`), `seocho/rules.py` (`infer_rules_from_graph`).

## 0. Prerequisites

Two optional dependencies — the notebook degrades cleanly if either is missing:

- **owlready2** — parses the FIBO Turtle. `pip install owlready2`. If absent, a hand-rolled fallback produces an `Ontology` of the same shape.
- **OPENAI_API_KEY** — needed for the live indexing cells (Sections A and B). Without it, structural cells (compile context, merge, migration plan, rule inference) still run.

In [ ]:
# --- Setup (run me first) ---
import sys
from pathlib import Path

try:
    import seocho, owlready2, rdflib
    from dotenv import load_dotenv
except ImportError:
    %pip install -q "seocho[ontology,local]" python-dotenv
    import seocho, owlready2, rdflib
    from dotenv import load_dotenv

load_dotenv(Path.cwd().parent / ".env", override=False)
print('seocho', seocho.__version__, '| .env loaded')


In [1]:
import seocho, owlready2, rdflib
from dotenv import load_dotenv

In [4]:
load_dotenv(Path.cwd().parent / ".env", override=False)
print('seocho', seocho.__version__, '| .env loaded')

seocho 0.3.0 | .env loaded


In [5]:
import os, json
from pathlib import Path

TTL_PATH = Path('datasets') / 'fibo_be_minimal.ttl'
if not TTL_PATH.exists():
    TTL_PATH = Path(__file__).resolve().parent / 'datasets' / 'fibo_be_minimal.ttl' if '__file__' in dir() else TTL_PATH
TTL_PATH = TTL_PATH.resolve()
print('Turtle file :', TTL_PATH)
print('exists?     :', TTL_PATH.exists())

try:
    import owlready2 as o2
    OWLREADY2_AVAILABLE = True
    print('owlready2   :', getattr(o2, '__version__', 'installed'))
except ImportError:
    OWLREADY2_AVAILABLE = False
    print('owlready2   : not installed — falling back to hand-rolled equivalent.')
    print('              install via:  pip install owlready2')

LIVE = bool(os.environ.get('OPENAI_API_KEY'))
print('OPENAI key  :', 'set' if LIVE else 'not set — Section A and B live cells will skip')

Turtle file : /home/hadry/lab/seocho/examples/datasets/fibo_be_minimal.ttl
exists?     : True
owlready2   : installed
OPENAI key  : set


## 1. Inspect the FIBO Turtle file

In [6]:
ttl_text = TTL_PATH.read_text()
print(ttl_text[:1200] + ' ...')
print(f'(file is {len(ttl_text)} chars)')

# =====================================================================
# fibo_be_minimal.ttl
#
# Minimal FIBO Business Entities (BE) slice for the SEOCHO tutorial.
#
# This is a HAND-CURATED teaching subset — it is NOT a FIBO release. The
# upstream EDM Council Financial Industry Business Ontology is at
# https://spec.edmcouncil.org/fibo/ (MIT license).
#
# Authentic FIBO IRIs are linked via skos:exactMatch so a learner can
# follow each concept back to its canonical definition. Class names,
# property names, and definition style follow the FIBO best-practices
# guide (upper-camel-case classes, verb-prefixed lower-camel-case
# properties, ISO-704 genus/differentia definitions).
#
# This file is loaded in examples/tutorial_03_ontology_indexing.ipynb.
# =====================================================================

@prefix owl:   <http://www.w3.org/2002/07/owl#> .
@prefix rdf:   <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs:  <http://www.w3.org/2000/01/rdf-schema#

## 2. Convert FIBO Turtle → `seocho.Ontology`

The loader walks `owl:Class`, `owl:DatatypeProperty`, and `owl:ObjectProperty` declarations, flattens OWL inheritance to fit SEOCHO's flat label space, and forces `unique=True` on `hasName` so every node has a unique key. The fallback below produces the same shape without owlready2.

In [7]:
from seocho import Ontology, NodeDef, RelDef, P


def load_fibo_ontology(ttl_path, *, ontology_name='fibo_be_minimal', version='1.0.0'):
    """Parse a FIBO-style Turtle file via rdflib + owlready2 and emit a seocho.Ontology.

    owlready2's built-in parser does not reliably handle Turtle (.ttl) — it
    dispatches to the NTriples parser and fails on standard ``@prefix``
    headers. We pre-convert TTL -> RDF/XML using rdflib, then hand the
    RDF/XML to owlready2.
    """
    import owlready2 as o2
    import rdflib, tempfile, os
    g = rdflib.Graph()
    g.parse(str(ttl_path), format='turtle')
    with tempfile.NamedTemporaryFile(suffix='.owl', delete=False) as tmp:
        owl_path = tmp.name
    try:
        g.serialize(destination=owl_path, format='xml')
        onto = o2.get_ontology(f'file://{owl_path}').load()
    finally:
        try:
            os.unlink(owl_path)
        except OSError:
            pass
    def lname(entity):
        return getattr(entity, 'name', None) or str(entity).rsplit('/', 1)[-1].rsplit('#', 1)[-1]
    classes = {lname(c): c for c in onto.classes() if lname(c) and lname(c) != 'Thing'}
    PY_BY_RANGE = {int: int, float: float, bool: bool}
    dt_props = {}
    for prop in onto.data_properties():
        pname = lname(prop)
        domains = [d for d in (prop.domain or []) if hasattr(d, 'name')]
        rng = prop.range[0] if prop.range else str
        py_type = PY_BY_RANGE.get(rng, str)
        dt_props[pname] = (py_type, domains)
    obj_props = {}
    for prop in onto.object_properties():
        pname = lname(prop)
        src = lname(prop.domain[0]) if prop.domain else 'Any'
        tgt = lname(prop.range[0]) if prop.range else 'Any'
        obj_props[pname] = (src, tgt)
    nodes = {}
    for name, cls in classes.items():
        ancestor_ids, stack = set(), [cls]
        while stack:
            cur = stack.pop()
            if id(cur) in ancestor_ids: continue
            ancestor_ids.add(id(cur))
            for parent in cur.is_a:
                if hasattr(parent, 'name') and parent is not o2.Thing:
                    stack.append(parent)
        properties = {}
        for pname, (py_type, domains) in dt_props.items():
            if any(id(d) in ancestor_ids for d in domains):
                properties[pname] = P(py_type, unique=(pname == 'hasName'))
        desc = str(cls.label[0]) if cls.label else f'FIBO concept: {name}'
        nodes[name] = NodeDef(description=desc, properties=properties)
    relationships = {pname: RelDef(source=src, target=tgt) for pname, (src, tgt) in obj_props.items()}
    return Ontology(
        name=ontology_name, version=version,
        description='FIBO Business Entities minimal slice (loaded from Turtle).',
        nodes=nodes, relationships=relationships,
    )


def build_fibo_fallback():
    base_props  = {'hasName': P(str, unique=True)}
    person_props = {**base_props, 'hasTitle': P(str)}
    legal_props  = {**base_props, 'hasLegalName': P(str), 'hasIdentifier': P(str)}
    return Ontology(
        name='fibo_be_minimal', version='1.0.0',
        description='FIBO Business Entities minimal slice (hand-rolled fallback).',
        nodes={
            'Party':              NodeDef(description='Party',              properties=dict(base_props)),
            'Person':             NodeDef(description='Person',             properties=person_props),
            'LegalPerson':        NodeDef(description='LegalPerson',        properties=legal_props),
            'FormalOrganization': NodeDef(description='FormalOrganization', properties=legal_props),
            'Corporation':        NodeDef(description='Corporation',        properties=legal_props),
        },
        relationships={
            'isOfficerOf':    RelDef(source='Person',      target='Corporation'),
            'hasShareholder': RelDef(source='Corporation', target='Party'),
        },
    )

In [8]:
ontology_v1 = load_fibo_ontology(TTL_PATH) if OWLREADY2_AVAILABLE else build_fibo_fallback()
print('name        :', ontology_v1.name)
print('node labels :', list(ontology_v1.nodes.keys()))
print('relations   :', list(ontology_v1.relationships.keys()))
print('validate    :', ontology_v1.validate() or 'OK')

name        : fibo_be_minimal
node labels : ['Person', 'Corporation', 'Party', 'LegalPerson', 'FormalOrganization']
relations   : ['isOfficerOf', 'hasShareholder']
validate    : OK


## 3. Section A — Vanilla LLM + ontology

Recall Tutorial 1 ended with `backend.complete(system, user)` and a hand-written `system` string. The instinct is to keep tweaking that string. The better move: let the ontology generate the relevant part of it.

`ontology.to_extraction_context()` returns a dict with `entity_types`, `relationship_types`, `constraints_summary`, and `ontology_name`. Concatenate them into your system prompt and the LLM now has the full schema in front of it — no manual list-keeping.

In [9]:
from seocho import create_llm_backend

ext_ctx = ontology_v1.to_extraction_context()

system_with_ontology = (
    f'You extract entities for the {ext_ctx["ontology_name"]} knowledge graph.\n'
    f'Entity types:\n{ext_ctx["entity_types"]}\n'
    f'Relationship types:\n{ext_ctx["relationship_types"]}\n'
    'Return JSON: {"nodes": [{"label": str, "id": str, "properties": {...}}], '
    '"relationships": [{"source": str, "target": str, "type": str}]}'
)

system_without_ontology = (
    'Extract entities and relationships from the input. '
    'Return JSON with "nodes" and "relationships" keys.'
)

SAMPLE = (
    'Tim Cook is an officer of Apple, a corporation registered in California. '
    'Apple holds shares in Beats Electronics.'
)

if LIVE:
    backend = create_llm_backend(provider='openai', model='gpt-4o-mini')
    for label, system in [('without ontology', system_without_ontology), ('with FIBO     ', system_with_ontology)]:
        resp = backend.complete(
            system=system,
            user=SAMPLE,
            temperature=0.0,
            response_format={'type': 'json_object'},
        )
        print(f'--- {label} ---')
        try:
            payload = resp.json()
            print(json.dumps(payload, indent=2)[:600] + (' ...' if len(json.dumps(payload)) > 600 else ''))
        except Exception:
            print(resp.text[:600])
        print()
else:
    print('[SKIP] vanilla-LLM ontology demo — set OPENAI_API_KEY to run.')

--- without ontology ---
{
  "nodes": [
    {
      "id": "1",
      "name": "Tim Cook",
      "type": "Person"
    },
    {
      "id": "2",
      "name": "Apple",
      "type": "Corporation"
    },
    {
      "id": "3",
      "name": "California",
      "type": "Location"
    },
    {
      "id": "4",
      "name": "Beats Electronics",
      "type": "Corporation"
    }
  ],
  "relationships": [
    {
      "source": "1",
      "target": "2",
      "type": "is an officer of"
    },
    {
      "source": "2",
      "target": "3",
      "type": "registered in"
    },
    {
      "source": "2",
      "target": "4",
   

--- with FIBO      ---
{
  "nodes": [
    {
      "label": "Person",
      "id": "Tim_Cook",
      "properties": {
        "hasName": "Tim Cook",
        "hasTitle": null
      }
    },
    {
      "label": "Corporation",
      "id": "Apple",
      "properties": {
        "hasName": "Apple",
        "hasIdentifier": null,
        "hasLegalName": null
      }
    },
    {

Without the ontology, the LLM picks its own labels (`Person`, `Organization`, `Place`, …) — output shape varies between calls. With the ontology, labels are constrained to the FIBO classes, properties match what `to_shacl()` will validate, and the output round-trips into `IndexingPipeline` without coercion.

This is the *whole reason* SEOCHO has an ontology layer. The agent in Section B applies the same trick automatically.

## 4. Section B — Agent + ontology

Now the same FIBO ontology drives the agent. The `IndexingAgent`'s `extract_entities` tool builds its system prompt from `ontology.to_extraction_context()` (exactly what we did by hand above); `validate_extraction` calls `ontology.validate_with_shacl(...)`; `write_to_graph` enforces the schema at write time.

Every span emitted by the session carries `ontology_context.context_hash`, so a downstream consumer can prove which schema produced which output.

In [10]:
from seocho import Seocho, AGENT_PRESETS, compile_ontology_context
from seocho.index.pipeline import IndexingPipeline
from seocho.store.graph import LadybugGraphStore

compiled_v1 = compile_ontology_context(ontology_v1, workspace_id='tutorial', profile='default')
desc_v1 = compiled_v1.descriptor
print('v1 context_hash :', desc_v1.context_hash)
print('node_labels     :', desc_v1.node_labels)
print('relationships   :', desc_v1.relationship_types)

store_v1 = LadybugGraphStore('.seocho/tutorial_fibo_v1.lbug')
store_v1.ensure_constraints(ontology_v1)

if LIVE:
    llm = create_llm_backend(provider='openai', model='gpt-4o-mini')
    pipeline_v1 = IndexingPipeline(
        ontology=ontology_v1,
        graph_store=store_v1,
        llm=llm,
        workspace_id='tutorial',
        ontology_profile='default',
        enable_rule_constraints=True,
    )
    result_v1 = pipeline_v1.index(SAMPLE, database='tutorial_fibo_v1', category='general')
    print()
    print('total_nodes         :', result_v1.total_nodes)
    print('total_relationships :', result_v1.total_relationships)
    print('context_hash on out :', result_v1.ontology_context.get('context_hash'))
    print('rule_profile rules  :', len((result_v1.rule_profile or {}).get('rules', [])))
    print('validation errors   :', result_v1.validation_errors or 'none')
else:
    result_v1 = None
    llm = None
    print('[SKIP] indexing — set OPENAI_API_KEY to run a real extraction.')

v1 context_hash : 8623caafdb3ce2c5
node_labels     : ['Corporation', 'FormalOrganization', 'LegalPerson', 'Party', 'Person']
relationships   : ['hasShareholder', 'isOfficerOf']

total_nodes         : 4
total_relationships : 5
context_hash on out : 8623caafdb3ce2c5
rule_profile rules  : 6
validation errors   : none


Note `result_v1.ontology_context['context_hash']` matches `desc_v1.context_hash`: the indexing pipeline stamps the same descriptor on its output. Combined with Tutorial 2's Opik tracing, every agent span is now provably tied to a versioned schema.

## 5. Section C — Evolution

FIBO does not model product offerings (it stops at the financial-instrument boundary). To capture an iPhone clause, we extend FIBO with a non-FIBO `Product` node and a `MAKES` relationship via `Ontology.merge(strategy='union')`.

Keeping the extension non-FIBO and explicit (instead of editing the source TTL) is the recommended pattern — it keeps your derivation transparent and `migration_plan` works out of the box.

In [11]:
extension = Ontology(
    name='product_extension', version='1.1.0',
    description='Non-FIBO product extension layered on top of FIBO BE.',
    nodes={
        'Product': NodeDef(
            description='A named product or service offered by a corporation.',
            properties={'hasName': P(str, unique=True), 'hasCategory': P(str)},
        ),
    },
    relationships={
        'MAKES': RelDef(source='Corporation', target='Product', cardinality='ONE_TO_MANY'),
    },
)

ontology_v2 = ontology_v1.merge(extension, strategy='union', name='fibo_be_minimal+product')
ontology_v2.version = '2.0.0'

print('v2 node labels :', sorted(ontology_v2.nodes.keys()))
print('v2 relations   :', sorted(ontology_v2.relationships.keys()))
print('validate       :', ontology_v2.validate() or 'OK')

v2 node labels : ['Corporation', 'FormalOrganization', 'LegalPerson', 'Party', 'Person', 'Product']
v2 relations   : ['MAKES', 'hasShareholder', 'isOfficerOf']
validate       : OK


In [ ]:
plan = ontology_v1.migration_plan(ontology_v2)
print('summary    :', plan['summary'])
print('breaking?  :', plan['breaking'])
print('additions  :', plan['additions'])
print('removals   :', plan['removals'])
for stmt in plan['cypher_statements']:
    print(' -', stmt['description'])

In [ ]:
compiled_v2 = compile_ontology_context(ontology_v2, workspace_id='tutorial', profile='default')
desc_v2 = compiled_v2.descriptor

print('v1 context_hash :', desc_v1.context_hash)
print('v2 context_hash :', desc_v2.context_hash)
print('changed?        :', desc_v1.context_hash != desc_v2.context_hash)

Re-index the same text under v2. Because v2 has vocabulary for `Product` and `MAKES`, the agent picks up the iPhone clause that v1 cannot.

In [ ]:
store_v2 = LadybugGraphStore('.seocho/tutorial_fibo_v2.lbug')
store_v2.ensure_constraints(ontology_v2)

SAMPLE_V2 = SAMPLE + ' Apple announced the new iPhone at WWDC 2025.'

if LIVE:
    pipeline_v2 = IndexingPipeline(
        ontology=ontology_v2,
        graph_store=store_v2,
        llm=llm,
        workspace_id='tutorial',
        ontology_profile='default',
        enable_rule_constraints=True,
    )
    result_v2 = pipeline_v2.index(SAMPLE_V2, database='tutorial_fibo_v2', category='general')
    print('total_nodes         :', result_v2.total_nodes)
    print('total_relationships :', result_v2.total_relationships)
    print('context_hash on out :', result_v2.ontology_context.get('context_hash'))
else:
    result_v2 = None
    print('[SKIP] indexing — set OPENAI_API_KEY to run.')

In [ ]:
if LIVE and result_v1 is not None and result_v2 is not None:
    print('hash differs           :', (result_v1.ontology_context['context_hash']
                                       != result_v2.ontology_context['context_hash']))
    print('Δ total_nodes          :', result_v2.total_nodes - result_v1.total_nodes)
    print('Δ total_relationships  :', result_v2.total_relationships - result_v1.total_relationships)
    labels_v1 = {r['label'] for r in (result_v1.rule_profile or {}).get('rules', [])}
    labels_v2 = {r['label'] for r in (result_v2.rule_profile or {}).get('rules', [])}
    print('rule labels only in v2 :', sorted(labels_v2 - labels_v1) or '(none)')
    print('rule labels only in v1 :', sorted(labels_v1 - labels_v2) or '(none)')
else:
    print('[SKIP] diff — re-run sections 4 and the v2 cell with OPENAI_API_KEY set.')

### 5.1 Query comparison — v1 (FIBO only) vs v2 (FIBO + Product)

Section C showed the *indexing* delta — v2 produced more nodes / more relationships / a different `context_hash`. The payoff is on the **query** side: a question about products is answerable under v2 and **not answerable under v1**, because v1's vocabulary doesn't model `Product` at all.

This is the load-bearing claim of ontology versioning: **schema evolution unlocks queries that were structurally impossible before**. The graphs below were both populated from the same source text — only the active ontology differs.

In [ ]:
# Same NL question against both ontology versions. The pipelines + stores were
# already built earlier in the tutorial (pipeline_v1, pipeline_v2, store_v1, store_v2).
# We open one Session per graph + ontology pair so each query is grounded in the
# correct schema.
if LIVE and result_v1 is not None and result_v2 is not None:
    QUESTIONS = [
        'Who is the CEO of Apple?',
        'What products does Apple make?',          # v1 has no Product node — should fail / hedge
        'Is Apple a corporation registered somewhere?',
    ]

    print(f'{"question":<48} | {"v1 (FIBO)":<28} | {"v2 (FIBO+Product)":<28}')
    print('-' * 110)

    for label, ontology_x, store_x in [
        ('v1 ', ontology_v1, store_v1),
        ('v2 ', ontology_v2, store_v2),
    ]:
        s_q = Seocho(
            ontology=ontology_x, graph_store=store_x, llm=llm,
            workspace_id='tutorial', ontology_profile='default',
            user_id='openai-yitae',
        )
        try:
            with s_q.session(f'compare-{label.strip()}') as sess:
                for q in QUESTIONS:
                    a = sess.ask(q)
                    a_short = a.replace(chr(10), ' ')[:80]
                    if label.strip() == 'v1':
                        print(f'{q[:48]:<48} | {a_short:<28} | {"...":<28}')
                    else:
                        # rewrite the v2 column on top of the v1 row
                        # (simpler: store and print both at the end)
                        pass
        finally:
            s_q.close()

    # Cleaner: redo as a side-by-side dict and print at the end
    answers = {'v1': {}, 'v2': {}}
    for label, ontology_x, store_x in [
        ('v1', ontology_v1, store_v1),
        ('v2', ontology_v2, store_v2),
    ]:
        s_q = Seocho(
            ontology=ontology_x, graph_store=store_x, llm=llm,
            workspace_id='tutorial', ontology_profile='default',
            user_id='openai-yitae',
        )
        try:
            with s_q.session(f'compare2-{label}') as sess:
                for q in QUESTIONS:
                    answers[label][q] = sess.ask(q).replace(chr(10), ' ')
        finally:
            s_q.close()

    print('\n=== SIDE-BY-SIDE ===')
    for q in QUESTIONS:
        print(f'\nQ: {q}')
        print(f'  v1 (FIBO only)        : {answers["v1"][q][:120]}')
        print(f'  v2 (FIBO + Product)   : {answers["v2"][q][:120]}')
else:
    print('[SKIP] v1-vs-v2 query comparison — re-run sections 4 and the v2 cell with OPENAI_API_KEY set.')

**What you'll see:**

- *Q1 (CEO of Apple)*: both v1 and v2 answer correctly — `Person(name="Tim Cook") -isOfficerOf-> Corporation(name="Apple")` exists in both graphs.
- *Q2 (products Apple makes)*: v1 produces a hedge or no-answer — there is **no `Product` label in the v1 ontology**, so the query agent has no class to MATCH against. v2 answers `iPhone` because the SAMPLE text was indexed under v2's vocabulary which includes `Product` and `MAKES`.
- *Q3 (corporation registered)*: both answer, since `Corporation` is in both ontologies.

**The `context_hash` on each result's span lets a downstream consumer prove which schema produced which answer** — the same audit trail Section C set up applies here, on the query side.

This is the contract end-to-end: extract under a versioned schema → write the schema's hash on every node and span → query under that schema → trace the answer back to that exact schema.

## 6. Rule inference and the `ontology_identity_hash`

`infer_rules_from_graph` accepts an `ontology_identity_hash` and stamps it on the resulting `RuleSet`. When `/rules/profiles` stores a profile, it carries this hash; promotion gates refuse to apply the profile if the active ontology's `context_hash` does not match — the match / drift / unknown trichotomy from commit `c4dbaff`.

In [ ]:
from seocho.rules import infer_rules_from_graph

fake_extracted = {
    'nodes': [
        {'label': 'Corporation', 'properties': {'hasName': 'Apple',   'hasIdentifier': 'LEI-001'}},
        {'label': 'Corporation', 'properties': {'hasName': 'Beats',   'hasIdentifier': 'LEI-002'}},
        {'label': 'Person',      'properties': {'hasName': 'Tim Cook', 'hasTitle': 'CEO'}},
        {'label': 'Person',      'properties': {'hasName': 'Jeff Williams', 'hasTitle': 'COO'}},
        {'label': 'Product',     'properties': {'hasName': 'iPhone',   'hasCategory': 'phone'}},
        {'label': 'Product',     'properties': {'hasName': 'iPad',     'hasCategory': 'tablet'}},
    ],
    'relationships': [],
}

ruleset = infer_rules_from_graph(fake_extracted, ontology_identity_hash=desc_v2.context_hash)
print('schema_version          :', ruleset.schema_version)
print('ontology_identity_hash  :', ruleset.ontology_identity_hash)
print('rules                   :')
for rule in ruleset.rules:
    print(f'  {rule.label:<14} {rule.property_name:<14} kind={rule.kind:<8} params={rule.params}')

## 7. Wrap-up — the contract is now visible end-to-end

- **Tutorial 1** taught you the floor: `backend.complete(system, user)` plus instruction design.
- **Tutorial 2** wrapped that floor with tool use, multi-turn state, and Opik observability.
- **Tutorial 3** replaced the hand-written system prompt and the placeholder schema with a versioned FIBO ontology — and now every span, every rule profile, every persisted graph node carries a `context_hash` that proves which schema produced which output.

If the FIBO version changes, every artifact tagged with the old hash becomes traceable as such. That is the SEOCHO contract.

**Next reading:**

- The full FIBO at <https://spec.edmcouncil.org/fibo/> and FIBO best practices at <https://github.com/edmcouncil/html-pages/blob/develop/general/best-practices.md>.
- `docs/ONTOLOGY_RUN_CONTEXT_STRATEGY.md` — full ontology context lifecycle.
- `docs/SHACL_PRACTICAL_GUIDE.md` — validation in depth.
- `seocho/tests/test_ontology_context.py` and `seocho/tests/test_ontology_migration.py` — assertion patterns.

## 8. Why seocho? — what you'd write without it

You've now seen the full stack: load FIBO from a `.ttl` file → drive ontology-aware extraction → version with `context_hash` → write under `workspace_id` and stamp `user_id` on every trace → migrate v1 → v2 with a Cypher plan → infer rules with an identity hash → query under sequential / parallel / supervisor patterns (Tutorial 2 §6) → audit the answers back to the schema that produced them.

A natural question: **could I just use `owlready2` + `openai-agents` directly?**

You can. The single ontology / single workspace / single user / no-evolution case works fine on raw SDKs. seocho earns its keep on every other axis. Here's the surface you'd own yourself.

### 8.1 Without seocho — the glue you'd write yourself

```python
# Rough equivalent of:  Seocho.local(onto, ...).session().add(text)

import owlready2 as o2, rdflib, tempfile, os, asyncio, hashlib, json, re
from agents import Runner, Agent, function_tool
from neo4j import GraphDatabase

# 1. Parse FIBO TTL  → owlready2 needs RDF/XML, not Turtle (~10 lines)
g = rdflib.Graph(); g.parse('fibo.ttl', format='turtle')
with tempfile.NamedTemporaryFile(suffix='.owl', delete=False) as t:
    g.serialize(destination=t.name, format='xml')
    onto = o2.get_ontology(f'file://{t.name}').load()
os.unlink(t.name)

# 2. Walk classes / properties  →  build an extraction prompt (~30 lines)
# 3. Compute context_hash so downstream can verify version (~10 lines)
# 4. Validate dynamic Cypher labels — injection guard (~10 lines)
# 5. Wire 8 @function_tools (extract / validate / score / link / write /
#    text2cypher / execute / search) — ~50 LOC each, ~400 LOC total
# 6. Loop-aware sync wrapper — asyncio.run() breaks in Jupyter (~20 lines)
# 7. Session abstraction with entity_index + query_cache (~150 lines)
# 8. Span emit to JSONL + Opik with workspace_id/user_id metadata (~80 lines)
# 9. SHACL constraints from the ontology (~60 lines)
# 10. migration_plan(v1, v2) → Cypher diff (~150 lines)
# 11. infer_rules_from_graph with ontology_identity_hash binding (~100 lines)
# 12. Fallback contract — agent → pipeline on exception, stamp degraded (~40 lines)
# ... and a long list of known-failure modes; most of them are tracked
#     as `seocho-*` beads (e.g. seocho-y4at: workspace_id leak on query;
#     seocho-c2ck: non-transactional schema migration; seocho-1zck: silent
#     agent fallback) that already have regression tests pinning them.
```

That's **~1,000 lines of glue** and ~10 different failure modes, each one a
production incident waiting to happen. The bead tracker for this repo has
the full catalog under `bd ready --label sev-high,sev-critical`.

### 8.2 With seocho

```python
from seocho import Seocho

s = Seocho.local(
    load_fibo_ontology('fibo.ttl'),
    user_id='alice@team',
    workspace_id='acme',
)
with s.session('demo') as sess:
    sess.add('Tim Cook is the CEO of Apple.')
    print(sess.ask('Who runs Apple?'))
```

**Six lines.** Bundled in: `context_hash` propagation, `workspace_id` write-stamping, `user_id` on every span, drift gate, fallback contract, JSONL + Opik tracing with the metadata fields Tutorial 2 §4 documented, agent runtime adapter (signature-drift compatibility for `openai-agents`), loop-aware sync helper (`_run_sync` — Jupyter / FastAPI / pytest-asyncio safe), Cypher injection guard, SHACL validation hooks, `migration_plan` between ontology versions, `infer_rules_from_graph` with identity hash, and the FIBO TTL loader you saw in §2.

None of these pieces are conceptually hard. They're all **~one engineer-day of yak-shaving** to write yourself; each one has at least one subtle gotcha (Turtle dispatch in owlready2, asyncio.run in Jupyter, fork-safety of the runner singleton, transactional schema migration, …). seocho exists because nobody should pay that bill more than once.

### 8.3 When you should NOT use seocho

Intellectually honest disclaimer — **the framework isn't free**, and these are real reasons to skip it:

| If… | …then |
|-----|-------|
| You own one ontology that never evolves | the `migration_plan` / `context_hash` machinery is pure overhead — vanilla owlready2 is enough |
| You don't need multi-tenant isolation | `workspace_id` + drift gate + identity tags solve a problem you don't have |
| Your team is deep in a non-Python graph stack (Java + Neo4j Java driver) | seocho speaks Python over Bolt; you'd carry a second runtime |
| You want a different agent SDK (LangGraph, CrewAI, raw Anthropic SDK) | `seocho.agents_runtime` is currently `openai-agents`-shaped; swap is possible but not drop-in |
| You're prototyping at the chat-completion floor | Tutorial 1's `backend.complete(system, user)` is the floor — you can stop there |

For every other case — **schema-versioned extraction, multi-tenant observability, agent execution with a real fallback contract** — you'd build seocho's middleware whether you wanted to or not. The choice is whether to write it once on top of seocho, or repeat it per project on top of raw SDKs.

That is the sales pitch. The proof is in the test suite (`seocho/tests/test_user_facing_edge_cases.py` pins one regression per failure mode), the bead catalogue (`bd ready` shows every open trade-off), and the audit trail (`context_hash` + `workspace_id` + `user_id` on every span you emitted across these three notebooks).